# Exposure Analysis – Quickstart Notebook

## Purpose of this notebook

This notebook provides a **simple and interactive way to run an exposure analysis**.

It is intended for users who want to:
- run an exposure analysis step by step,
- explore intermediate results interactively,

---

## How this notebook is intended to be used

This notebook is designed to be executed **cell by cell**, from top to bottom.

Each section performs a specific task:
- loading inputs,
- running exposure processing,
- visualizing results.

Users are encouraged to:
- run cells sequentially,
- stop and inspect outputs,
- re-run individual cells if needed.

---

## Configuration

This notebook uses the **project configuration file (`config.yaml`)**.

Before running the notebook:
- make sure the configuration file is correctly set,
- verify that input paths and parameters are valid.

No code modification is required in this notebook.

---
## Before you start

Please make sure that:
- the `config.yaml` file is correctly configured
- all input paths exist
- required hazards are activated in the configuration

Once this is done, you can run the notebook cells from top to bottom.




## 1. Load configuration and inputs

In [ ]:

# --- Config ---
import sys
import os
sys.path.append(os.path.abspath(".."))
from modules.config_utils import load_config
CONFIG_PATH = "../config.yml"
CONFIG = load_config(str(CONFIG_PATH))
print("Output directory:", CONFIG.get("output_dir"))

In [ ]:
# --- Load AOI and infrastructure (by type) ---
import geopandas as gpd
import pandas as pd
from modules.crs_utils import harmonize_crs

# AOI
aoi = gpd.read_file(CONFIG["aoi"])
aoi_union = aoi.union_all()

# Infrastructure
points_by_type = {}
lines_by_type = {}

for name, path in CONFIG["infrastructure_inputs"]["points"].items():
    if path is not None:
        points_by_type[name] = gpd.read_file(path)

for name, path in CONFIG["infrastructure_inputs"]["lines"].items():
    if path is not None:
        lines_by_type[name] = gpd.read_file(path)

# Harmonize to AOI CRS + clip + tag
for name, gdf in points_by_type.items():
    gdf, = harmonize_crs([gdf], aoi.crs)
    gdf = gdf[gdf.geometry.within(aoi_union)]
    gdf["infra_type"] = name
    points_by_type[name] = gdf

for name, gdf in lines_by_type.items():
    gdf, = harmonize_crs([gdf], aoi.crs)
    gdf = gdf[gdf.geometry.intersects(aoi_union)]
    gdf["infra_type"] = name
    lines_by_type[name] = gdf

# Union for quick previews
all_points = gpd.GeoDataFrame(pd.concat(points_by_type.values(), ignore_index=True), crs=aoi.crs) if points_by_type else None
all_lines  = gpd.GeoDataFrame(pd.concat(lines_by_type.values(),  ignore_index=True), crs=aoi.crs) if lines_by_type else None

print({k: len(v) for k, v in points_by_type.items()})
print({k: len(v) for k, v in lines_by_type.items()})

In [ ]:
# --- Show initial network map (by type) ---
from modules.plotting import plot_initial_map_by_type
from pathlib import Path
out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

initial_map_path = out_dir / "00_initial_network_map.png"
plot_initial_map_by_type(
    aoi=aoi,
    points_by_type=points_by_type,
    lines_by_type=lines_by_type,
    output_path=str(initial_map_path),
)
print("Saved:", initial_map_path)


## 2. Run exposure analysis

In [ ]:
# --- Run raster-based exposure (pluvial_flood / fluvial_flood / landslide / earthquake, etc.) ---
from modules.raster_exposure import process_raster_exposures

# Number of samples along each line to test exposure
SAMPLE_POINTS_PER_LINE = int(CONFIG.get("sample_points_per_line", 10))

results = process_raster_exposures(
    config=CONFIG,
    aoi=aoi,
    points_by_type=points_by_type,
    lines_by_type=lines_by_type,
    sample_points_per_line=SAMPLE_POINTS_PER_LINE
)
print("Available hazard results:", list(results.keys()))

## 3. Visualize results

In [ ]:
# --- Quick preview: show the first few exposed features for each hazard (if any) ---
import pandas as pd

def head_if_any(gdf, n=5):
    if gdf is None or len(gdf) == 0:
        return pd.DataFrame({"note": ["<empty>"]})
    return gdf.head(n).drop(columns=[c for c in gdf.columns if c.startswith("geometry")], errors="ignore")

for hz, payload in results.items():
    print(f"\n### {hz} ###")
    pts = payload.get("points_exposed")
    lns = payload.get("lines_exposed")
    print("Points:")
    display(head_if_any(pts))
    print("Lines:")
    display(head_if_any(lns))

## 4. Check generated outputs

## End of the quickstart

If no error message is displayed above, the exposure analysis has completed successfully.

You can now:
- review generated maps,
- use output files in GIS software,
- re-run the notebook if needed.
